In [1]:
# ============================================================
# 04b_joint_retrain_AB.ipynb
# Joint retrain on A+B after B-domain transfer fine-tuning
# Finalized pipeline:
#   01 -> 02 -> 03(source A) -> 04(finetune B) -> 04b(joint A+B) -> 05
# ============================================================

from pathlib import Path
import json
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras

# -----------------------------
# Config
# -----------------------------
DATA_ROOT = Path("/home/tonyliao/Location")
BUILD_DIR    = DATA_ROOT / "dataset_build_hybrid"
TRANSFER_DIR = DATA_ROOT / "transfer_finetune_B_runs"
OUT_DIR      = DATA_ROOT / "joint_retrain_AB_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 64
EPOCHS_STAGE1 = 60
EPOCHS_STAGE2 = 100

# smaller than stage 04
LR_STAGE1 = 5e-5
LR_STAGE2 = 1e-5

TARGET_T = 256
TARGET_S = None

TRANSFER_MODEL_PATH = TRANSFER_DIR / "hybrid_transfer_B_best.keras"

# Stage 1: mostly heads / light adaptation
FREEZE_KEYWORDS_STAGE1 = [
    "amp_ssl_encoder",
    "amp_encoder_local",
    "phase_geometry_encoder",
]

# Always frozen
ALWAYS_FROZEN_LAYER_NAMES = [
    "coarse_xy_out",
]

# -----------------------------
# Load data
# -----------------------------
def load_npz(path: Path):
    obj = np.load(path, allow_pickle=True)
    return {k: obj[k] for k in obj.files}

AB_joint_retrain = load_npz(BUILD_DIR / "AB_joint_retrain.npz")
AB_joint_val     = load_npz(BUILD_DIR / "AB_joint_val.npz")

with open(BUILD_DIR / "label_map.json", "r", encoding="utf-8") as f:
    label_meta = json.load(f)

label_map = label_meta["label_map"]
inv_label_map = {int(k): v for k, v in label_meta["inv_label_map"].items()}
num_classes = len(label_map)

print("AB_joint_retrain:", len(AB_joint_retrain["amp_path"]))
print("AB_joint_val:", len(AB_joint_val["amp_path"]))
print("num_classes:", num_classes)

if len(AB_joint_retrain["amp_path"]) == 0:
    raise ValueError("AB_joint_retrain is empty.")
if len(AB_joint_val["amp_path"]) == 0:
    raise ValueError("AB_joint_val is empty.")
if not TRANSFER_MODEL_PATH.exists():
    raise FileNotFoundError(f"Missing transfer model: {TRANSFER_MODEL_PATH}")

# -----------------------------
# Coordinates
# -----------------------------
CLASS_CENTER_MAP = {
    "Empty": [0.0, 0.0],
    # "Center": [3.0, 4.0],
    # "Corner": [0.0, 8.0],
    "Corner_LeftDown": [1.0, 7.0],
    "Corner_LeftUp": [1.0, 1.0],
    "Corner_RightDown": [5.0, 7.0],
    "Corner_RightUp": [5.0, 1.0],
    # "Empty_Pred": [0.0, 0.0],
    "LeftDown": [2.0, 6.0],
    # "LeftDown_Far": [1.5, 7.0],
    "LeftMid": [2.0, 4.0],
    "LeftUp": [2.0, 2.0],
    # "LeftUp_Near": [2.0, 2.5],
    # "LeftUp_Pred": [2.0, 2.0],
    "MiddleDown": [3.0, 6.0],
    "MiddleUp": [3.0, 2.0],
    "RightDown": [4.0, 6.0],
    "RightMid": [4.0, 4.0],
    "RightUp": [4.0, 2.0],
    # "Wall": [6.0, 5.0],
}

missing_centers = sorted([
    lab for lab in label_map.keys()
    if lab not in CLASS_CENTER_MAP
])
if missing_centers:
    raise ValueError(
        f"These labels exist in label_map but not in CLASS_CENTER_MAP: {missing_centers}"
    )

# -----------------------------
# Helpers
# -----------------------------
def ensure_3d(x):
    x = np.asarray(x, dtype=np.float32)
    if x.ndim == 2:
        x = x[..., None]
    return x

def resize_to_target(x, target_t, target_s):
    x_tf = tf.convert_to_tensor(x, dtype=tf.float32)
    x_tf = tf.image.resize(x_tf, size=(target_t, target_s), method="bilinear")
    return x_tf.numpy().astype(np.float32)

def zscore_per_sample(x):
    mu = np.mean(x, axis=(0, 1), keepdims=True)
    sd = np.std(x, axis=(0, 1), keepdims=True) + 1e-6
    return ((x - mu) / sd).astype(np.float32)

def load_amp(path):
    x = np.load(str(path)).astype(np.float32)
    return ensure_3d(x)

def load_pha(path, ref_shape=None):
    if path is None or str(path) == "":
        if ref_shape is None:
            raise ValueError("pha_path missing and ref_shape is None")
        return np.zeros(ref_shape, dtype=np.float32)

    p = Path(str(path))
    if not p.exists():
        if ref_shape is None:
            raise ValueError(f"pha_path does not exist: {path}")
        return np.zeros(ref_shape, dtype=np.float32)

    x = np.load(str(p)).astype(np.float32)
    return ensure_3d(x)

sample_amp = load_amp(AB_joint_retrain["amp_path"][0])
if TARGET_S is None:
    TARGET_S = sample_amp.shape[1]

print("TARGET_T =", TARGET_T)
print("TARGET_S =", TARGET_S)

def preprocess_amp(path):
    x = load_amp(path)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def preprocess_pha(path, amp_shape=None):
    x = load_pha(path, ref_shape=amp_shape)
    x = resize_to_target(x, TARGET_T, TARGET_S)
    x = zscore_per_sample(x)
    return x.astype(np.float32)

def onehot(num_classes, y):
    v = np.zeros((num_classes,), dtype=np.float32)
    v[int(y)] = 1.0
    return v

def xy_from_label(label_name):
    label_name = str(label_name)
    if label_name not in CLASS_CENTER_MAP:
        raise KeyError(f"Label {label_name} not found in CLASS_CENTER_MAP")
    return np.asarray(CLASS_CENTER_MAP[label_name], dtype=np.float32)

# -----------------------------
# Dataset sequence
# -----------------------------
class HybridSequence(keras.utils.Sequence):
    def __init__(self, obj, batch_size=16, shuffle=True):
        self.obj = obj
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indices = np.arange(len(obj["amp_path"]))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        inds = self.indices[idx*self.batch_size:(idx+1)*self.batch_size]

        x_amp = []
        x_pha = []

        y_presence = []
        y_class = []
        y_coarse = []
        y_delta = []
        y_final = []

        sw_presence = []
        sw_class = []
        sw_coarse = []
        sw_delta = []
        sw_final = []

        for i in inds:
            amp = preprocess_amp(self.obj["amp_path"][i])
            pha = preprocess_pha(self.obj["pha_path"][i], amp_shape=amp.shape)

            label_name = str(self.obj["anchor_label"][i])
            label_id   = int(self.obj["label_id"][i])
            presence   = int(self.obj["presence"][i])

            if presence == 1:
                xy = xy_from_label(label_name)
            else:
                xy = np.zeros((2,), dtype=np.float32)

            coarse_xy = xy.copy()
            delta_xy  = np.zeros((2,), dtype=np.float32)

            x_amp.append(amp)
            x_pha.append(pha)

            y_presence.append([presence])
            y_class.append(onehot(num_classes, label_id))
            y_coarse.append(coarse_xy)
            y_delta.append(delta_xy)
            y_final.append(xy)

            sw_presence.append(1.0)
            sw_class.append(1.0)

            loc_w = 1.0 if presence == 1 else 0.0
            sw_coarse.append(loc_w)
            sw_delta.append(loc_w)
            sw_final.append(loc_w)

        x = {
            "amp_in": np.stack(x_amp).astype(np.float32),
            "pha_in": np.stack(x_pha).astype(np.float32),
        }

        y = {
            "presence_out": np.stack(y_presence).astype(np.float32),
            "class_out": np.stack(y_class).astype(np.float32),
            "coarse_xy_out": np.stack(y_coarse).astype(np.float32),
            "amp_delta_out": np.stack(y_delta).astype(np.float32),
            "final_xy_out": np.stack(y_final).astype(np.float32),
        }

        sw = {
            "presence_out": np.asarray(sw_presence, dtype=np.float32),
            "class_out": np.asarray(sw_class, dtype=np.float32),
            "coarse_xy_out": np.asarray(sw_coarse, dtype=np.float32),
            "amp_delta_out": np.asarray(sw_delta, dtype=np.float32),
            "final_xy_out": np.asarray(sw_final, dtype=np.float32),
        }

        return x, y, sw

train_seq = HybridSequence(AB_joint_retrain, batch_size=BATCH_SIZE, shuffle=True)
val_seq   = HybridSequence(AB_joint_val, batch_size=BATCH_SIZE, shuffle=False)

bx, by, bsw = train_seq[0]
print("amp_in:", bx["amp_in"].shape)
print("pha_in:", bx["pha_in"].shape)
for k, v in by.items():
    print(k, v.shape)

# -----------------------------
# Load transfer-finetuned model
# -----------------------------
model = keras.models.load_model(TRANSFER_MODEL_PATH, compile=False)
print("Loaded model from:", TRANSFER_MODEL_PATH)
model.summary()

# -----------------------------
# Loss / metrics
# -----------------------------
losses = {
    "presence_out": keras.losses.BinaryCrossentropy(),
    "class_out": keras.losses.CategoricalCrossentropy(label_smoothing=0.02),
    "coarse_xy_out": keras.losses.Huber(delta=0.5),
    "amp_delta_out": keras.losses.Huber(delta=0.25),
    "final_xy_out": keras.losses.Huber(delta=0.5),
}

loss_weights = {
    "presence_out": 1.0,
    "class_out": 1.0,
    "coarse_xy_out": 1.0,
    "amp_delta_out": 0.25,
    "final_xy_out": 1.5,
}

metrics = {
    "presence_out": [
        keras.metrics.BinaryAccuracy(name="acc"),
        keras.metrics.Precision(name="prec"),
        keras.metrics.Recall(name="rec"),
    ],
    "class_out": [
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.TopKCategoricalAccuracy(k=2, name="top2"),
    ],
    "coarse_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "amp_delta_out": [keras.metrics.MeanAbsoluteError(name="mae")],
    "final_xy_out": [keras.metrics.MeanAbsoluteError(name="mae")],
}

def compile_model(model, lr):
    model.compile(
        optimizer=keras.optimizers.Adam(lr),
        loss=losses,
        loss_weights=loss_weights,
        metrics=metrics,
    )

# -----------------------------
# Trainability control
# -----------------------------
def set_stage1_trainable(model):
    for layer in model.layers:
        layer.trainable = True

        lname = layer.name.lower()
        if any(key.lower() in lname for key in FREEZE_KEYWORDS_STAGE1):
            layer.trainable = False

        if layer.name in ALWAYS_FROZEN_LAYER_NAMES:
            layer.trainable = False

def set_stage2_trainable(model):
    for layer in model.layers:
        layer.trainable = True
        if layer.name in ALWAYS_FROZEN_LAYER_NAMES:
            layer.trainable = False

def summarize_trainable_layers(model, title):
    print(f"\n{title}")
    total = 0
    trainable = 0
    for layer in model.layers:
        total += 1
        if layer.trainable:
            trainable += 1
        print(f"{layer.name:35s} trainable={layer.trainable}")
    print(f"Trainable layers: {trainable}/{total}")

# -----------------------------
# Stage 1 joint retrain
# -----------------------------
set_stage1_trainable(model)
summarize_trainable_layers(model, "04b Stage 1 trainable layers")
compile_model(model, LR_STAGE1)

callbacks_stage1 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_joint_AB_stage1_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "joint_AB_stage1_log.csv")),
]

history_stage1 = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS_STAGE1,
    callbacks=callbacks_stage1,
    verbose=1,
)

model.save(OUT_DIR / "hybrid_joint_AB_stage1_final.keras")

# -----------------------------
# Stage 2 gentle global retrain
# -----------------------------
set_stage2_trainable(model)
summarize_trainable_layers(model, "04b Stage 2 trainable layers")
compile_model(model, LR_STAGE2)

callbacks_stage2 = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(OUT_DIR / "hybrid_joint_AB_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        mode="min",
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUT_DIR / "joint_AB_stage2_log.csv")),
]

history_stage2 = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS_STAGE2,
    callbacks=callbacks_stage2,
    verbose=1,
)

model.save(OUT_DIR / "hybrid_joint_AB_final.keras")

# -----------------------------
# Save history / summary
# -----------------------------
history_all = {
    "stage1": history_stage1.history,
    "stage2": history_stage2.history,
}

with open(OUT_DIR / "joint_AB_history.json", "w", encoding="utf-8") as f:
    json.dump(history_all, f, indent=2)

receiver_counts_train = {}
if "receiver_domain" in AB_joint_retrain:
    uniq, cnt = np.unique(AB_joint_retrain["receiver_domain"].astype(str), return_counts=True)
    receiver_counts_train = {str(k): int(v) for k, v in zip(uniq, cnt)}

receiver_counts_val = {}
if "receiver_domain" in AB_joint_val:
    uniq, cnt = np.unique(AB_joint_val["receiver_domain"].astype(str), return_counts=True)
    receiver_counts_val = {str(k): int(v) for k, v in zip(uniq, cnt)}

summary = {
    "training_domain": "A_plus_B_joint_retrain",
    "base_model_path": str(TRANSFER_MODEL_PATH),
    "single_receiver_input_only": True,
    "fused_AB_input": False,
    "joint_retrain_protocol": "two_stage",
    "uses_transfer_model_from_B": True,
    "used_target_eval_unseen_locations": False,
    "stage1": {
        "freeze_keywords": FREEZE_KEYWORDS_STAGE1,
        "always_frozen_layer_names": ALWAYS_FROZEN_LAYER_NAMES,
        "epochs": EPOCHS_STAGE1,
        "learning_rate": LR_STAGE1,
        "monitor": "val_loss",
    },
    "stage2": {
        "always_frozen_layer_names": ALWAYS_FROZEN_LAYER_NAMES,
        "epochs": EPOCHS_STAGE2,
        "learning_rate": LR_STAGE2,
        "monitor": "val_loss",
    },
    "target_t": TARGET_T,
    "target_s": int(TARGET_S),
    "AB_joint_retrain_windows": int(len(AB_joint_retrain["amp_path"])),
    "AB_joint_val_windows": int(len(AB_joint_val["amp_path"])),
    "receiver_counts_train": receiver_counts_train,
    "receiver_counts_val": receiver_counts_val,
    "num_classes": int(num_classes),
}

with open(OUT_DIR / "joint_AB_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("Saved to:", OUT_DIR)
print(json.dumps(summary, indent=2))

2026-04-13 14:41:16.704388: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-13 14:41:16.710831: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776091276.718328 1601638 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776091276.720583 1601638 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776091276.726418 1601638 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

AB_joint_retrain: 6060
AB_joint_val: 1513
num_classes: 13
TARGET_T = 256
TARGET_S = 41


I0000 00:00:1776091277.941459 1601638 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 22181 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4090, pci bus id: 0000:01:00.0, compute capability: 8.9


amp_in: (64, 256, 41, 2)
pha_in: (64, 256, 41, 2)
presence_out (64, 1)
class_out (64, 13)
coarse_xy_out (64, 2)
amp_delta_out (64, 2)
final_xy_out (64, 2)
Loaded model from: /home/tonyliao/Location/transfer_finetune_B_runs/hybrid_transfer_B_best.keras


Model: "hybrid_source_A_model"

...


Epoch 1: val_loss improved from inf to 0.52000, saving model to /home/tonyliao/Location/joint_retrain_AB_runs/hybrid_joint_AB_stage1_best.keras
95/95 ━━━━━━━━━━━━━━━━━━━━ 31s 241ms/step - amp_delta_out_loss: 0.0047 - amp_delta_out_mae: 0.0815 - class_out_acc: 0.8233 - class_out_loss: 0.6037 - class_out_top2: 0.9219 - coarse_xy_out_loss: 0.1267 - coarse_xy_out_mae: 0.4140 - final_xy_out_loss: 0.1267 - final_xy_out_mae: 0.4142 - loss: 1.0453 - presence_out_acc: 0.9687 - presence_out_loss: 0.1235 - presence_out_prec: 0.9674 - presence_out_rec: 0.9999 - val_amp_delta_out_loss: 0.0035 - val_amp_delta_out_mae: 0.0728 - val_class_out_acc: 0.9610 - val_class_out_loss: 0.3216 - val_class_out_top2: 0.9901 - val_coarse_xy_out_loss: 0.0574 - val_coarse_xy_out_mae: 0.2405 - val_final_xy_out_loss: 0.0573 - val_final_xy_out_mae: 0.2397 - val_loss: 0.5200 - val_presence_out_acc: 0.9808 - val_presence_out_loss: 0.0490 - val_presence_out_prec: 0.9817 - val_presence_out_rec: 0.9978 - learning_rate: 5.00

2026-04-13 14:53:38.401266: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 256 bytes spill stores, 256 bytes spill loads

2026-04-13 14:53:38.458463: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 68 bytes spill stores, 68 bytes spill loads

2026-04-13 14:53:38.629060: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 408 bytes spill stores, 408 bytes spill loads

2026-04-13 14:53:38.635646: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 408 bytes spill stores, 408 bytes spill loads

2026-04-13 14:53:38.963549: I 

18/95 ━━━━━━━━━━━━━━━━━━━━ 8s 112ms/step - amp_delta_out_loss: 9.6528e-04 - amp_delta_out_mae: 0.0469 - class_out_acc: 1.0000 - class_out_loss: 0.1520 - class_out_top2: 1.0000 - coarse_xy_out_loss: 0.0020 - coarse_xy_out_mae: 0.0512 - final_xy_out_loss: 0.0020 - final_xy_out_mae: 0.0501 - loss: 0.1577 - presence_out_acc: 1.0000 - presence_out_loss: 4.7853e-04 - presence_out_prec: 1.0000 - presence_out_rec: 1.0000

2026-04-13 14:53:47.900977: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11191', 68 bytes spill stores, 68 bytes spill loads

2026-04-13 14:53:47.997560: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 104 bytes spill stores, 104 bytes spill loads

2026-04-13 14:53:48.018167: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 480 bytes spill stores, 480 bytes spill loads

2026-04-13 14:53:48.031530: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_11189', 484 bytes spill stores, 484 bytes spill loads

2026-04-13 14:53:48.033163: I 

95/95 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step - amp_delta_out_loss: 9.8925e-04 - amp_delta_out_mae: 0.0518 - class_out_acc: 1.0000 - class_out_loss: 0.1514 - class_out_top2: 1.0000 - coarse_xy_out_loss: 0.0014 - coarse_xy_out_mae: 0.0419 - final_xy_out_loss: 0.0014 - final_xy_out_mae: 0.0413 - loss: 0.1557 - presence_out_acc: 1.0000 - presence_out_loss: 4.6395e-04 - presence_out_prec: 1.0000 - presence_out_rec: 1.0000
Epoch 1: val_loss improved from inf to 0.16850, saving model to /home/tonyliao/Location/joint_retrain_AB_runs/hybrid_joint_AB_best.keras
95/95 ━━━━━━━━━━━━━━━━━━━━ 34s 248ms/step - amp_delta_out_loss: 9.8918e-04 - amp_delta_out_mae: 0.0518 - class_out_acc: 1.0000 - class_out_loss: 0.1514 - class_out_top2: 1.0000 - coarse_xy_out_loss: 0.0014 - coarse_xy_out_mae: 0.0418 - final_xy_out_loss: 0.0014 - final_xy_out_mae: 0.0412 - loss: 0.1557 - presence_out_acc: 1.0000 - presence_out_loss: 4.6594e-04 - presence_out_prec: 1.0000 - presence_out_rec: 1.0000 - val_amp_delta_out_loss: 6.